# Track 1 iron reconstruction
Image-conditioned geometry from an automatically semantically verified crop. No tripod candidate, manual labels or reference geometry.

In [ ]:
import os,sys,subprocess,json
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run(['uv','python','install','3.11'],check=True)
subprocess.run(['uv','venv','--python','3.11','/tmp/shape-env'],check=True)
py='/tmp/shape-env/bin/python'
subprocess.run(['uv','pip','install','--python',py,'torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run(['uv','pip','install','--python',py,'transformers==4.48.3','diffusers==0.32.2','accelerate','einops','omegaconf','trimesh','pillow','numpy<2','scikit-image','opencv-python<4.12','pyyaml','tqdm','safetensors','huggingface_hub<1','pygltflib','pymeshlab','fast_simplification','ninja','pybind11'],check=True)
repo=Path('/tmp/Hunyuan3D-2')
subprocess.run(['git','clone','--depth','1','https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git',str(repo)],check=True)
revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
subprocess.run(['uv','pip','install','--python',py,'--no-deps','-e',str(repo)],check=True)
env=os.environ.copy();env['PYTHONPATH']=str(repo)


In [ ]:
import hashlib
ranking=json.loads(next(Path('/kaggle/input').rglob('semantic_crop_ranking.json')).read_text())
by_hash={x['input_sha256']:x for x in ranking}
candidates=[]
for path in Path('/kaggle/input').rglob('object_observation.png'):
 if path.parent.name not in {'episode_000006','episode_000007','episode_000008'}:continue
 report=json.loads(path.with_name('segmentation_report.json').read_text())
 if report.get('semantic_selection'):
  margin=float(report['seed']['semantic_margin'])
 else:
  value=by_hash[hashlib.sha256(path.read_bytes()).hexdigest()]
  margin=float(value['semantic_margin'])
 if margin>0:candidates.append((margin,path,report))
assert candidates,'No automatic iron crop passes semantic verification'
margin,image,segmentation=max(candidates,key=lambda item:item[0])
print('Automatic iron source',image,'semantic margin',margin,flush=True)


In [ ]:
Path('/tmp/reconstruct_object.py').write_text('"""Reconstruct an object from its automatic RGBA video crop using Hunyuan3D.\n\nThis produces a normalized shape hypothesis. Metric scale and camera registration\nmust be estimated from the Track 1 observations before using it in a submission.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom huggingface_hub import HfApi, snapshot_download\nfrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--image\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--weights\',type=Path,required=True)\n    p.add_argument(\'--revision\',help=\'Pin a previously recorded HF model revision\')\n    p.add_argument(\'--steps\',type=int,default=50)\n    p.add_argument(\'--resolution\',type=int,default=384)\n    p.add_argument(\'--seed\',type=int,default=42)\n    a=p.parse_args()\n    if a.steps<1 or a.resolution<64:p.error(\'Invalid sampling settings\')\n    image=Image.open(a.image).convert(\'RGBA\')\n    alpha=np.asarray(image)[:,:,3]\n    if not (alpha>0).any() or (alpha>0).all():\n        raise ValueError(\'Expected an actual masked RGBA observation\')\n    repo=\'tencent/Hunyuan3D-2\'\n    revision=a.revision or HfApi().model_info(repo).sha\n    snapshot_download(repo,revision=revision,local_dir=a.weights,\n        allow_patterns=[\'hunyuan3d-dit-v2-0/**\',\'hunyuan3d-vae-v2-0/**\',\'*.json\'])\n    pipeline=Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(str(a.weights),\n        subfolder=\'hunyuan3d-dit-v2-0\',device=\'cuda\',dtype=torch.float16)\n    with torch.inference_mode():\n        mesh=pipeline(image=image,num_inference_steps=a.steps,\n            octree_resolution=a.resolution,num_chunks=4000,\n            generator=torch.Generator(device=\'cuda\').manual_seed(a.seed))[0]\n    if not len(mesh.vertices) or not len(mesh.faces) or not np.isfinite(mesh.vertices).all():\n        raise ValueError(\'Image-conditioned model produced invalid geometry\')\n    a.output.mkdir(parents=True,exist_ok=True)\n    mesh.export(a.output/\'normalized_shape.glb\')\n    report={\'model\':repo,\'revision\':revision,\'seed\':a.seed,\'steps\':a.steps,\n        \'resolution\':a.resolution,\'input_sha256\':hashlib.sha256(a.image.read_bytes()).hexdigest(),\n        \'vertices\':len(mesh.vertices),\'faces\':len(mesh.faces),\n        \'metric_scale_estimated\':False,\'camera_registration_estimated\':False,\n        \'ready_for_submission\':False,\'manual_geometry_used\':False}\n    (a.output/\'shape_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n')
Path('/tmp/prepare_mesh.py').write_text('"""Remove degenerate generation debris and fit a real mesh to the scoring budget."""\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport numpy as np\nimport trimesh\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--mesh\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--strict-topology\',action=\'store_true\')\n    a=p.parse_args()\n    mesh=trimesh.load(a.mesh,force=\'mesh\',process=False)\n    if not isinstance(mesh,trimesh.Trimesh) or not np.isfinite(mesh.vertices).all():\n        raise ValueError(\'Expected finite triangle geometry\')\n    original={\'vertices\':len(mesh.vertices),\'faces\':len(mesh.faces)}\n    mesh.process(validate=True)\n    if mesh.area<=0 or not len(mesh.faces):raise ValueError(\'Degenerate object geometry\')\n    parts=mesh.split(only_watertight=False)\n    threshold=mesh.area*1e-6\n    retained=[x for x in parts if x.area>=threshold]\n    if not retained:raise ValueError(\'No nondegenerate object component\')\n    mesh=trimesh.util.concatenate(retained)\n    before={\'components\':len(retained),\'watertight\':mesh.is_watertight,\'euler_number\':int(mesh.euler_number)}\n    if len(mesh.faces)>4096 or len(mesh.vertices)>4096:\n        mesh=mesh.simplify_quadric_decimation(face_count=4096)\n        mesh.remove_unreferenced_vertices()\n        mesh.process(validate=True)\n        if before[\'watertight\'] and not mesh.is_watertight:mesh.fill_holes()\n    if len(mesh.faces)>4096 or len(mesh.vertices)>4096:\n        raise ValueError(\'Simplification did not meet the official mesh budget\')\n    if not np.isfinite(mesh.vertices).all() or mesh.area<=0:\n        raise ValueError(\'Simplification produced invalid geometry\')\n    warnings=[]\n    if before[\'watertight\'] and not mesh.is_watertight:warnings.append(\'Budget simplification opened surface boundaries\')\n    if int(mesh.euler_number)!=before[\'euler_number\']:warnings.append(\'Budget simplification changed Euler characteristic\')\n    if a.strict_topology and warnings:raise ValueError(\'; \'.join(warnings))\n    a.output.mkdir(parents=True,exist_ok=True)\n    path=a.output/\'normalized_shape.glb\';mesh.export(path)\n    report={\'source_sha256\':hashlib.sha256(a.mesh.read_bytes()).hexdigest(),\n        \'mesh_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n        \'original\':original,\'before_simplification\':before,\n        \'vertices\':len(mesh.vertices),\'faces\':len(mesh.faces),\n        \'watertight\':mesh.is_watertight,\'euler_number\':int(mesh.euler_number),\n        \'removed_components\':len(parts)-len(retained),\'metric_scale_estimated\':False,\n        \'quality_warnings\':warnings,\'strict_topology\':a.strict_topology,\n        \'budget_method\':\'quadric edge collapse, matching the official fixed face-budget approach\'}\n    (a.output/\'mesh_preparation_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n')
subprocess.run([py,'/tmp/reconstruct_object.py','--image',str(image),'--output','/kaggle/working/raw_iron','--weights','/tmp/shape_weights'],env=env,check=True)
subprocess.run([py,'/tmp/prepare_mesh.py','--mesh','/kaggle/working/raw_iron/normalized_shape.glb','--output','/kaggle/working/iron'],env=env,check=True)
Path('/kaggle/working/iron_source.json').write_text(json.dumps({'source_episode':int(image.parent.name.split('_')[-1]),'source_sha256':hashlib.sha256(image.read_bytes()).hexdigest(),'semantic_margin':margin,'Hunyuan3D_code_revision':revision,'manual_annotations_used':False},indent=2))
